## Customer Support Chatbot

In [ ]:


# Import standard libraries for file handling and text processing
import os, pathlib, textwrap, glob

# Load documents from various sources (URLs, text files, PDFs)
from langchain_community.document_loaders import UnstructuredURLLoader, TextLoader, PyPDFLoader

# Split long texts into smaller, manageable chunks for embedding
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Vector store to store and retrieve embeddings efficiently using FAISS
from langchain.vectorstores import FAISS

# Generate text embeddings using OpenAI or Hugging Face models
from langchain.embeddings import OpenAIEmbeddings, HuggingFaceEmbeddings, SentenceTransformerEmbeddings

# Use local LLMs (e.g., via Ollama) for response generation
from langchain.llms import Ollama

# Build a retrieval chain that combines a retriever, a prompt, and an LLM
from langchain.chains import ConversationalRetrievalChain

# Create prompts for the RAG system
from langchain.prompts import PromptTemplate

print("✅ Libraries imported! You're good to go!")



### 1 - Data preparation

The goal of this step is to turn all reference documents into small chunks of text that a retriever can index and search. These documents typically come from:

    PDF files: local documents such as policies, user manuals, or guides.
    Web pages (HTML): online documentation, blog posts, or help articles.

In this step, we perform two actions:

    Ingesting: load every PDF and collect the raw text in a list named raw_docs.
    Chunking: split each document into small, overlapping chunks so later steps can match a user query to the most relevant passage.

    Step 1. Load the pdfs and get the data
    Step 2. Data chunking
    Step 3. Embedding creation of chunks and indexing




## 2 - Data preparation

The goal of this step is to turn all reference documents into small chunks of text that a retriever can index and search. These documents typically come from:

    PDF files: local documents such as policies, user manuals, or guides.
    Web pages (HTML): online documentation, blog posts, or help articles.

In this step, we perform two actions:

    Ingesting: load every PDF and collect the raw text in a list named raw_docs.
    Chunking: split each document into small, overlapping chunks so later steps can match a user query to the most relevant passage.




### 2.1 - Ingest source documents

We can use different libraries to load and process PDFs. A quick web search will show several options, each with its own strengths. In this case, we’ll use PyPDFLoader from LangChain, which makes it easy to extract text from PDF files for downstream processing. To learn more about how to use it, refer to: https://python.langchain.com/docs/integrations/document_loaders/pypdfloader/

Use PyPDFLoader to load every PDF whose filename matches data/Everstorm_*.pdf and collect all pages in a list called raw_docs. The content of these PDFs is synthetically generated for educational purposes.


In [ ]:
def load_offline_files():
    pdf_paths = glob.glob("data/Everstorm_*.pdf")
    raw_docs = []

    for file_path in pdf_paths:
        try:
            loader = PyPDFLoader(file_path)
            docs = loader.load()

            # adding source to all doc in docs
            if docs:
                for doc in docs:
                    doc.metadata['source'] = file_path
                raw_docs.extend(docs)
        
        except Exception as e:
            print(f"Error loading documents {file_path} : {e}")
            continue

    print(f"Loaded {len(raw_docs)} PDF pages from {len(pdf_paths)} files.")
    return raw_docs

### 2.1 - Load web pages

You can also pull content straight from the web. Various libraries support reading and parsing web pages directly into text, which is useful for building custom knowledge bases. One example is UnstructuredURLLoader from LangChain, which can extract readable content from raw HTML pages and return them in a structured format.

To practice, load each HTML page below and store the results in a list called raw_docs. We’ve included a few sample URLs, but you can replace them with any links you prefer.

For robustness, add an offline fallback in case a URL fails. In real projects, we typically cache fetched pages to disk, handle rate limits, and track fetch timestamps so content can be refreshed periodically without relying on live network calls during development. For this project, we don’t have offline HTML copies available, but you can still practice by loading any PDFs from the data/ folder using PyPDFLoader and appending them to raw_docs.

raw_docs is a list of Document(a class defined in LangChain) objects. 
Example: [
    Document(page_content="...", metadata={...}),
    Document(page_content="...", metadata={...}),
]

In [ ]:
def load_files(file_paths: list):
    raw_doc = []

    for path_string in file_paths:
        pdf_path = glob.glob(f"data/{path_string}*.pdf")

        if not pdf_path:
            print(f"No PDF found matching: data/{path_string}*.pdf")
            continue

        try:
            loader = PyPDFLoader(pdf_path[0])
            docs = loader.load()

            if docs:
                for doc in docs:
                    doc.metadata["source"] = pdf_path[0]
                raw_doc.extend(docs)  # ← fixed: docs not doc

        except Exception as e:
            print(f"Error loading {pdf_path[0]}: {e}")

    return raw_doc


def load_fallback(url, url_mapping, raw_docs):
    if url not in url_mapping:
        print(f"No offline fallback configured for {url}")
        return
    offline_pdf = load_files(url_mapping[url])
    if offline_pdf:
        raw_docs.extend(offline_pdf)
    else:
        print(f"Unable to get data for {url}")


URLS = [
    "https://developer.bigcommerce.com/docs/store-operations/shipping",
    "https://developer.bigcommerce.com/docs/store-operations/orders/refunds",
]

url_mapping = {
    "https://developer.bigcommerce.com/docs/store-operations/shipping": ["Everstorm_Shipping", "Everstorm_Product"],
    "https://developer.bigcommerce.com/docs/store-operations/orders/refunds": ["Everstorm_Payment", "Everstorm_Return"],
}

raw_docs = []

for url in URLS:
    try:
        loader = UnstructuredURLLoader(urls=[url])
        docs = loader.load()

        valid_docs = [doc for doc in docs if doc.page_content.strip()]

        if valid_docs:
            for doc in valid_docs:
                doc.metadata["source"] = url
            raw_docs.extend(valid_docs)
        else:
            print(f"No useful content from {url}. Loading offline docs.")
            load_fallback(url, url_mapping, raw_docs)

    except Exception as e:
        print(f"⚠️  Web fetch failed for {url}: {e}")
        load_fallback(url, url_mapping, raw_docs)

print(f"\nTotal: {len(raw_docs)} documents loaded.")


### 2.2 - Chunk the text

Long documents won’t work well directly with most LLMs. They can easily exceed the model’s context window, making it impossible for the model to read or reason over the full text at once. Even if they fit, processing long inputs can be inefficient and lead to weaker retrieval results.

To handle this, we split large documents into smaller, overlapping chunks. Several libraries can help with text splitting, each designed to preserve structure or balance chunk size. A popular choice is RecursiveCharacterTextSplitter from LangChain, which splits text intelligently while keeping paragraph or sentence boundaries intact. To familiarize youself with the library, visit: https://python.langchain.com/api_reference/text_splitters/character/langchain_text_splitters.character.RecursiveCharacterTextSplitter.html

In this project, we’ll split each document into chunks of roughly 300 tokens with a 30-token overlap using RecursiveCharacterTextSplitter. This overlap helps maintain continuity across chunks while ensuring each piece stays small enough for embedding and retrieval.

